# EV Challenge — GoodWe | Sprint 04
### Pipeline de avaliação sistemática do agente em Python



> **Antes de começar:** tenha em mãos o pacote `Sprint04_GoodWe_Entrega.zip` e uma credencial OpenAI válida. **Colab → ícone de chave (Secrets) → `OPENAI_API_KEY`**, ative o acesso ao notebook. Não coloque a chave nas células nem no GitHub. Há uso pago da API na execução.
**Importante:** um ZIP executado pode conter rodada diferente das capturas de tela anteriores. A versão auditada prioriza `resultados/respostas_por_caso.csv` e recalcula o resultado a partir dele. A célula 5 **não consome API novamente** enquanto já houver 56 registros válidos, salvo se a opção `EXECUTAR_NOVAMENTE` for alterada para `True`.


## 1. Carregar os arquivos do projeto

Se o repositório já estiver aberto/clonado na pasta `/content/Sprint04_GoodWe`, não é necessário enviar nada. Caso contrário, carregue o arquivo **ZIP do projeto** quando o Colab solicitar. O conteúdo da Sprint 03 está incluído nele.

In [ ]:
from pathlib import Path
import os, sys, zipfile

PROJETO = Path('/content/Sprint04_GoodWe') if Path('/content').exists() else Path.cwd()
# O Colab pode conservar a pasta antiga da Sprint 04 entre execucoes.
# Confira a revisao do pipeline: so o notebook novo NAO substitui src/juiz.py.
VERSAO_ESPERADA = '2026-10-20-colab-pdf-fontes-portaveis-v5'
marcador = PROJETO / 'VERSAO_PIPELINE.txt'
versao_encontrada = marcador.read_text(encoding='utf-8').strip() if marcador.exists() else ''
projeto_atualizado = (PROJETO / 'data' / 'golden_dataset.json').exists() and versao_encontrada == VERSAO_ESPERADA
if not projeto_atualizado:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Abra o projeto na pasta Sprint04_GoodWe ou execute este notebook no Google Colab.') from exc
    print('Envie o ZIP atualizado Sprint04_GoodWe_COLAB_CORRIGIDO.zip:')
    enviados = files.upload()
    zips = [nome for nome in enviados if nome.lower().endswith('.zip')]
    if len(zips) != 1:
        raise RuntimeError('Envie exatamente um ZIP do projeto atualizado.')
    destino = Path('/content')
    with zipfile.ZipFile(zips[0]) as pacote:
        # ZIP produzido para este projeto e enviado pelo próprio usuário.
        for nome in pacote.namelist():
            caminho = Path(nome)
            if caminho.is_absolute() or '..' in caminho.parts:
                raise RuntimeError('O ZIP contém um caminho não permitido: ' + nome)
        pacote.extractall(destino)
    PROJETO = destino / 'Sprint04_GoodWe'
    import importlib
    for nome_modulo in list(sys.modules):
        if nome_modulo == 'src' or nome_modulo.startswith('src.'):
            del sys.modules[nome_modulo]
    importlib.invalidate_caches()

assert (PROJETO / 'VERSAO_PIPELINE.txt').read_text(encoding='utf-8').strip() == VERSAO_ESPERADA, 'Codigo Python desatualizado. Envie o ZIP corrigido.'
assert (PROJETO / 'referencias' / 'sprint03_goodwe_agentes_final.ipynb').exists(), 'A Sprint 03 precisa estar na pasta referencias.'
assert (PROJETO / 'data' / 'golden_dataset.json').exists(), 'Golden dataset não encontrado.'
sys.path.insert(0, str(PROJETO))
os.chdir(PROJETO)
print('Projeto pronto:', PROJETO)


## 2. Instalar dependências

O adaptador importa o notebook original da Sprint 03 e suas bibliotecas. A instalação é necessária apenas uma vez por ambiente Colab.

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(PROJETO / 'requirements.txt')])
print('Dependências instaladas.')

## 3. Verificar os 28 casos e a fidelidade à Sprint 03

São **15 casos reaproveitados integralmente da Sprint 03** (F1–F6, M1–M3, S1–S6) + **13 novos casos** (F7–F11, B1–B6, S7–S8). O conjunto cobre funcionalidade, memória, segurança, prompt injection, aderência ao escopo e bordas. Gabaritos estão documentados no JSON.

In [ ]:
import json
import pandas as pd
from collections import Counter

golden = json.loads((PROJETO / 'data' / 'golden_dataset.json').read_text(encoding='utf-8'))
assert len(golden) == 28
assert len({caso['id'] for caso in golden}) == 28
print('Total:',len(golden),'| Reaproveitados Sprint 03:',sum(c['origem']=='reaproveitado_sprint03' for c in golden))
print('Categorias:',dict(Counter(c['categoria'] for c in golden)))
display(pd.DataFrame([{'ID':c['id'], 'Categoria':c['categoria'], 'Origem':c['origem'],
                      'Pergunta':c['pergunta'], 'Gabarito':c['gabarito']} for c in golden]))

## 4. Resultados da Sprint 03 — referência histórica

A tabela apresenta **resultados reais registrados na Sprint 03**, preservados para demonstrar a evolução do projeto. A nova avaliação sistemática da Sprint 04, com golden dataset e métricas por versão, é executada na próxima seção.

In [ ]:
from src.historico import gerar_tabelas_historicas

historico, casos_historicos = gerar_tabelas_historicas()
tabela_sprint03 = pd.DataFrame(historico).rename(columns={
    'versao': 'Versão',
    'casos_historicos': 'Casos avaliados',
    'nota_media_sprint03': 'Nota média (0-10)',
    'taxa_aprovacao_sprint03': 'Taxa de aprovação',
    'tokens_medios_sprint03': 'Tokens médios',
    'latencia_media_sprint03': 'Latência média (s)',
})
display(tabela_sprint03)

## 5. Executar a avaliação automatizada COMPLETA nas duas versões reais

Serão testados **28 casos por versão** com as mesmas perguntas, ordens, gabaritos, regras e juiz independente:

- `agente_t02`: LangGraph + GPT-4o-mini (temperature=0,2; top_p=0,9)
- `agente_t09`: o mesmo LangGraph + GPT-4o-mini (temperature=0,9; top_p=1,0)

O juiz GPT-4o-mini (temperature=0) avalia correção, escopo, relevância, faithfulness (quando há contexto recuperado) e recusa segura quando aplicável. A versão fica oculta para o juiz. Cada erro é registrado como ausente, sem nota inventada. **Esta célula exige API e pode gerar custos.**

In [ ]:
# Se o ZIP ja contem 56 respostas completas, NAO refazer chamadas pagas.
# Mude EXECUTAR_NOVAMENTE para True apenas quando desejar uma nova rodada.
from src.executar import executar
from src.metricas import escolher_melhor
import pandas as pd, json
EXECUTAR_NOVAMENTE = False
metadados_path = PROJETO / 'resultados' / 'metadados_execucao.json'
metricas_path = PROJETO / 'resultados' / 'metricas_por_versao.csv'
respostas_path = PROJETO / 'resultados' / 'respostas_por_caso.csv'
rodada_preservada = (metadados_path.exists() and metricas_path.exists() and respostas_path.exists()
                    and len(pd.read_csv(respostas_path)) == 56
                    and len(pd.read_csv(metricas_path)) == 2)
if rodada_preservada and not EXECUTAR_NOVAMENTE:
    metadados = json.loads(metadados_path.read_text(encoding='utf-8'))
    print('Usando 56 respostas arquivadas no ZIP; NENHUMA chamada nova a API.')
    print('Rodada no ZIP:', metadados.get('id_execucao'), 'Vencedor conforme metadados:',metadados.get('versao_recomendada'))
else:
    print('Iniciando nova rodada. Isso consumira API e sobrescrevera a rodada arquivada!')
    metadados = executar(['agente_t02','agente_t09'])
    print('Rodada conclusiva?', metadados['resultado_conclusivo'])


## 6. Ler métricas, respostas e justificativa da classificação

A decisão sobre a melhor versão se baseia em números da rodada nova. Exige **28 julgamentos completos por configuração**. As notas da Sprint 03 ficam separadas em arquivos de histórico.

In [ ]:
res_dir = PROJETO / 'resultados'
resumo = pd.read_csv(res_dir / 'metricas_por_versao.csv')
respostas = pd.read_csv(res_dir / 'respostas_por_caso.csv')
display(resumo)
display(respostas[['versao','id','categoria','correcao','escopo','relevancia',
                    'faithfulness','recusa_segura','nota_final','aprovado','status','justificativa_juiz']])
print('Consulte respostas_por_caso.csv para verificar as respostas integrais e a origem de cada nota.')

## 6.1. Confronto automatizado histórico da Sprint 03 x Sprint 04

Esta etapa **não reexecuta a API**: compara apenas os **15 casos originais** usando as notas automáticas que já foram obtidas nas duas sprints. Se `resultados/respostas_por_caso.csv` já foi gerado no Colab, cria uma tabela completa caso a caso e uma síntese por versão. **As notas históricas do juiz LLM não são notas humanas.**

Depois da comparação, o PDF é regenerado automaticamente com as evidências novas. Se faltar o CSV original da execução, o relatório apenas apresenta os valores conferidos nas capturas, devidamente identificados.

In [ ]:
from src.comparacao_sprints import comparar_automaticas
from src.relatorio import gerar_relatorio

linhas_pareadas, resumo_pareado = comparar_automaticas(PROJETO)
if linhas_pareadas:
    display(pd.DataFrame(resumo_pareado))
    display(pd.DataFrame(linhas_pareadas)[['versao','id','nota_automatica_sprint03', 'nota_automatica_sprint04', 'diferenca_s4_menos_s3', 'mesmo_resultado_aprovacao_7']])
else:
    print('CSV integral da nova execução ainda não está nesta pasta. Nada foi inventado; a análise permanece limitada às capturas.')
print('Relatório atualizado:', gerar_relatorio(PROJETO))

## 6.2. Segunda leitura crítica por IA e relatório consolidado (sem API)

Esta etapa lê as **respostas reais arquivadas no CSV**, produz uma análise qualitativa auxiliar de 36 respostas, aponta divergências e junta um apêndice ao PDF da Sprint 04. **As notas da segunda leitura são de IA, não de uma pessoa e não são avaliações históricas da Sprint 03.** O sistema confere o hash SHA-256 dos textos originais para não reaproveitar pareceres sobre respostas diferentes. Nenhuma nota original do juiz é substituída.

In [ ]:
from src.revisao_ia import consolidar, gerar_parecer
parecer_csv, pareceres = gerar_parecer(PROJETO)
pdf_consolidado = consolidar(PROJETO)
print('Respostas revisadas nesta segunda leitura IA:', len(pareceres))
print('CSV de auditoria:', parecer_csv)
print('PDF consolidado:', pdf_consolidado)
print('Nota: pareceres de IA não representam notas humanas da Sprint 03.')

## 7. Comparar com a avaliação manual feita na Sprint 03

**Importante:** os arquivos enviados documentam avaliação automática antiga (regras + LLM juiz), mas **não incluem notas atribuídas manualmente por pessoas**. É necessário preencher com evidências reais `data/avaliacao_manual_sprint03.csv` (nota humana de 0 a 10, avaliador e observação). Não confunda a nota do LLM juiz da Sprint 03 com avaliação humana. Se o grupo não tiver as notas humanas originais, isso permanecerá registrado como limitação honesta no PDF.

Depois de preencher o CSV no repositório/ambiente, execute esta célula para atualizar o confronto e o PDF.

In [ ]:
from src.executar import comparar_manual
from src.metricas import salvar_csv
from src.relatorio import gerar_relatorio
import csv
with (res_dir / 'respostas_por_caso.csv').open(newline='',encoding='utf-8-sig') as f:
    linhas_respostas=list(csv.DictReader(f))
for linha in linhas_respostas:
    try:
        linha['nota_final'] = float(linha['nota_final']) if linha.get('nota_final') else None
    except ValueError:
        linha['nota_final'] = None
confronto = comparar_manual(linhas_respostas)
salvar_csv(res_dir / 'confronto_manual_sprint03.csv', confronto)
display(pd.DataFrame(confronto))
pdf = gerar_relatorio(PROJETO)
print('Relatório PDF atualizado:', pdf)
# Recria o consolidado depois do relatório principal para evitar PDF desatualizado.
from src.revisao_ia import consolidar
pdf_final = consolidar(PROJETO)
print('Relatório CONSOLIDADO atualizado:', pdf_final)


## 8. Baixar o PDF e os resultados

Entregar **o repositório público no GitHub** com este notebook, os arquivos `src/`, o golden dataset, a comparação das duas versões e o PDF atualizado. O grupo deve realizar commits próprios e regulares, sem nenhuma credencial no Git. A publicação e o envio do link à FIAP são realizados pelos integrantes.

**Entregável recomendado:** `relatorios/Relatorio_Sprint04_GoodWe_CONSOLIDADO.pdf` junto com `resultados/auditoria_critica_ia_sprint04.csv`. O relatório original permanece preservado para auditoria.

In [ ]:
from pathlib import Path
import zipfile
pacote_saida = PROJETO.parent / 'Sprint04_GoodWe_Resultado_Executado.zip'
with zipfile.ZipFile(pacote_saida,'w',zipfile.ZIP_DEFLATED) as z:
    for arquivo in PROJETO.rglob('*'):
        rel=arquivo.relative_to(PROJETO)
        if (not arquivo.is_file() or any(p in {'__pycache__','.git','.ipynb_checkpoints'} for p in rel.parts)
            or arquivo.name == '.env' or arquivo.suffix == '.pyc'):
            continue
        z.write(arquivo,arcname=str(Path('Sprint04_GoodWe') / rel))
print('Pacote pronto:',pacote_saida)
try:
    from google.colab import files
    files.download(str(pacote_saida))
except ImportError:
    print('Arquivo disponível localmente:',pacote_saida)
